# Training Modules

## Step 1: Data Pre-Processing

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np

from popsim.ml.preprocess_utils import mask_to_largest_group_mask
from popsim.tests.fixtures import load_cmod_test_dataset

ds = load_cmod_test_dataset()

# Unit conversions.
R0 = 0.68  # Alcator C-Mod major radius in meters
ds["ip_MA"] = 1e-6 * np.abs(ds["ip"])
ds["Wmhd_MJ"] = 1e-6 * ds["Wmhd"]
ds["p_oh_MW"] = 1e-6 * ds["p_oh"]
ds["p_rad_MW"] = 1e-6 * ds["p_rad"]
ds["p_icrf_MW"] = 1e-6 * ds["p_icrf"]
ds["p_lh_MW"] = 1e-6 * ds["p_lh"]
ds["BT_mag"] = np.abs(ds["BT"])
ds["n_e_19"] = 1e-19 * ds["n_e"]
ds["epsilon"] = ds["a_minor"] / R0


state_vars = ["Wmhd_MJ"]
target_vars = ["Wmhd_MJ"]
param_vars = ["ip_MA", "p_oh_MW", "p_rad_MW", "p_icrf_MW", "p_lh_MW", "BT_mag", "n_e_19", "kappa_area", "epsilon"]

ds = ds[state_vars + target_vars + param_vars]

# TODO(allenw): figure out why for some reason, training on this shot yields NaNs.
ds = ds.drop_sel(shot=1160511013)

# Only keep data that are within bounds.
energy_bounds = (0.05, 0.5)
ds["energy_in_bounds"] = (energy_bounds[0] < ds["Wmhd_MJ"]) & (ds["Wmhd_MJ"] < energy_bounds[1])

prad_bounds = (0.0, 10.0)
ds["prad_in_bounds"] = (prad_bounds[0] < ds["p_rad_MW"]) & (ds["p_rad_MW"] < prad_bounds[1])

ne_in_bounds = (5.0, 100.0)
ds["ne_in_bounds"] = (ne_in_bounds[0] < ds["n_e_19"]) & (ds["n_e_19"] < ne_in_bounds[1])

ds["values_in_bounds"] = ds["energy_in_bounds"] & ds["prad_in_bounds"] & ds["ne_in_bounds"]


ds["values_in_bounds_largest_group"] = mask_to_largest_group_mask(ds["values_in_bounds"], episode_dim="shot", time_dim="time_slice")

ds = ds.where(ds["values_in_bounds_largest_group"], drop=True)

# Only keep long shots.
shot_lengths = ds["Wmhd_MJ"].count("time_slice")

ds = ds.where(shot_lengths > 500, drop=True)

## Constructing Dataloaders

In [ ]:
import jax

from popsim.ml import make_dataloader
from popsim.ml.split_utils import split_dataset_along_dim

train_ds, val_ds = split_dataset_along_dim(ds, fracs=(0.8, 0.2), dim="shot", key=jax.random.PRNGKey(42))

train_dl = make_dataloader(
    ds=train_ds,
    time_coord="time",
    episode_coord="shot",
    state_init_vars=state_vars,
    param_vars=param_vars,
    target_vars=target_vars,
    segment_length=500,
    segment_overlap=400,
    batch_size=512,
    shuffle=True,  # Randomize the order of samples during training.
)

val_dl = make_dataloader(
    ds=val_ds,
    time_coord="time",
    episode_coord="shot",
    state_init_vars=state_vars,
    param_vars=param_vars,
    target_vars=target_vars,
    segment_length=None,
    batch_size=512,
    shuffle=True,  # Randomize the order of samples during training.
)

## Defining a Power Balance Module

Let's consider the problem of predicting stored energy dynamics, with two different `taue` predictors:
1. A power law predictor
2. A bounded NN predictor

Below, we define a `PowerBalanceModule` and show you can create one with either of the two predictors.

In [ ]:
import chex
import equinox as eqx
import jax
import jax.numpy as jnp

from popsim import ModuleBase


def tanh_bound(x, min_val, max_val):
    offset = 0.5 * (min_val + max_val)
    scale = 0.5 * (max_val - min_val)
    return offset + scale * jnp.tanh(x)


@chex.dataclass
class TauePredictorInputs:
    plasma_current: float  # [MA]
    B0: float  # On axis magnetic field [T]
    ne19: float  # Line-averaged electron density [10^19 m^-3]
    p_absorbed_MW: float  # Absorbed power [MW]
    p_rad_MW: float  # Radiated power [MW]
    kappa: float  # Elongation [-]
    epsilon: float  # Inverse aspect ratio [-]


class BoundedNNPredictor(eqx.Module):
    nn: eqx.Module
    min_val: float = eqx.field(static=True)
    max_val: float = eqx.field(static=True)

    def __call__(self, inp: TauePredictorInputs) -> float:
        arr = jnp.array([inp.plasma_current, inp.B0, inp.ne19, inp.p_absorbed_MW, inp.p_rad_MW, inp.kappa, inp.epsilon])
        nn_out = self.nn(arr)
        bounded = tanh_bound(nn_out, self.min_val, self.max_val)
        return bounded.squeeze()

    @classmethod
    def create_default(cls):
        return cls(
            nn=eqx.nn.MLP(in_size=7, out_size=1, width_size=32, depth=2, key=jax.random.PRNGKey(42)),
            min_val=0.01,
            max_val=0.15,
        )


class PowerLawPredictor(eqx.Module):
    coeff: float
    powers: dict[str, float]

    def __init__(self, coeff: float, powers: dict[str, float]):
        self.coeff = coeff
        self.powers = powers

    def __call__(self, inp: TauePredictorInputs) -> float:
        out = self.coeff * (
            inp.plasma_current ** self.powers["alpha_I"]
            * inp.B0 ** self.powers["alpha_B"]
            * inp.ne19 ** self.powers["alpha_N"]
            * inp.p_absorbed_MW ** self.powers["alpha_P"]
            * R0 ** self.powers["alpha_R"]
            * inp.kappa ** self.powers["alpha_kappa"]
            * inp.epsilon ** self.powers["alpha_epsilon"]
        )
        return out.squeeze()

    @classmethod
    def create_ipb98(cls) -> "PowerLawPredictor":
        powers = {
            "alpha_I": jnp.array(0.93),
            "alpha_B": jnp.array(0.15),
            "alpha_N": jnp.array(0.41),
            "alpha_P": jnp.array(-0.69),
            "alpha_R": jnp.array(1.97),
            "alpha_kappa": jnp.array(0.78),
            "alpha_epsilon": jnp.array(0.58),
        }
        return cls(
            coeff=jnp.array(0.0562),
            powers=powers,
        )


@chex.dataclass
class PowerBalanceModule(ModuleBase):
    @chex.dataclass
    class Config:
        taue_predictor: eqx.Module

    @chex.dataclass
    class State:
        stored_energy: float

    @chex.dataclass
    class Params:
        taue_predictor_inputs: TauePredictorInputs
        sources_and_sinks: dict[str, float]

    @chex.dataclass
    class Output:
        stored_energy: float
        predicted_taue: float

    config: Config

    def __call__(self, state: "State", params: "Params") -> tuple[State, Output]:
        taue = self.config.taue_predictor(params.taue_predictor_inputs)

        stored_energy_dot = -state.stored_energy / taue + sum(params.sources_and_sinks.values())

        state_dot = PowerBalanceModule.State(stored_energy=stored_energy_dot)
        outputs = PowerBalanceModule.Output(stored_energy=state.stored_energy, predicted_taue=taue)
        return state_dot, outputs


module_power_law = PowerBalanceModule(config=PowerBalanceModule.Config(taue_predictor=PowerLawPredictor.create_ipb98()))
module_nn = PowerBalanceModule(config=PowerBalanceModule.Config(taue_predictor=BoundedNNPredictor.create_default()))

## Defining the Training Env and Trainers

In [ ]:
import chex
import optax
import wandb
from jaxtyping import ArrayLike

from popsim.ml import IntegralLoss, Trainer
from popsim.ml.envs import ModuleEvalEnv
from popsim.ml.loggers import WandbLogger


@chex.dataclass
class PowerBalanceEnv(ModuleEvalEnv):
    module: PowerBalanceModule

    def __init__(self, module: PowerBalanceModule):
        self.module = module

    @staticmethod
    def create_state(data: dict[str, ArrayLike]):
        return PowerBalanceModule.State(stored_energy=data["Wmhd_MJ"])

    @staticmethod
    def create_params(data: dict[str, ArrayLike]):
        return PowerBalanceModule.Params(
            taue_predictor_inputs=TauePredictorInputs(
                plasma_current=data["ip_MA"],
                B0=data["BT_mag"],
                ne19=data["n_e_19"],
                p_absorbed_MW=data["p_oh_MW"] + data["p_icrf_MW"] + data["p_lh_MW"],
                p_rad_MW=data["p_rad_MW"],
                kappa=data["kappa_area"],
                epsilon=data["epsilon"],
            ),
            sources_and_sinks={"p_abs": data["p_oh_MW"] + data["p_rad_MW"] + data["p_icrf_MW"] + data["p_lh_MW"]},
        )


def power_law_env_getter(env: PowerBalanceEnv):
    power_law = env.module.config.taue_predictor
    return (
        power_law.coeff,
        power_law.powers["alpha_I"],
        power_law.powers["alpha_N"],
        power_law.powers["alpha_P"],
        power_law.powers["alpha_kappa"],
        power_law.powers["alpha_epsilon"],
    )


def nn_env_getter(env: PowerBalanceEnv):
    nn = env.module.config.taue_predictor
    return nn


def loss_fn(predicted, targets):
    return optax.l2_loss(predicted.stored_energy, targets["Wmhd_MJ"])


env_power_law = PowerBalanceEnv(module=module_power_law)
env_nn = PowerBalanceEnv(module=module_nn)
loss_fn = IntegralLoss(loss_fn)

In [ ]:
import holoviews as hv
import numpy as np
import wandb
import xarray as xr

from popsim.ml.eval import eval_module_on_data
from popsim.tree_util import keypath_to_string


def make_error_plots(eval_fn_inputs):
    hv.extension("matplotlib")
    inp_ds = eval_fn_inputs.input_ds
    out_ds = eval_fn_inputs.output_ds
    combined_ds = xr.merge([inp_ds["Wmhd_MJ"], out_ds["output.stored_energy"]])

    abs_error = np.abs(combined_ds["Wmhd_MJ"] - combined_ds["output.stored_energy"])
    percent_error = 100.0 * abs_error / combined_ds["Wmhd_MJ"]
    prediction_horizon = combined_ds["time"] - combined_ds["time"].isel(time_slice_input=0)

    # Multiple y axes.
    # Create scatter plots for each metric
    scatter_abs = hv.Scatter((prediction_horizon, abs_error), kdims=["Prediction Horizon (s)"], vdims=["Absolute Error (MJ)"]).opts(
        color="blue", s=0.1, ylim=(0.0, 0.2)
    )

    scatter_percent = hv.Scatter((prediction_horizon, percent_error), kdims=["Prediction Horizon (s)"], vdims=["Percent Error"]).opts(
        color="blue", s=0.1, ylim=(0.0, 100.0)
    )

    combined = scatter_abs + scatter_percent
    combined = combined.opts(
        title="Validation Set Stored Energy Prediction Errors",
    )
    fig = hv.render(combined)
    return fig


def get_model_scalars(eval_fn_inputs):
    model = eval_fn_inputs.model

    path_and_leaves = jax.tree_util.tree_leaves_with_path(model)
    str_and_leaves = {
        keypath_to_string(keypath): leaf.item() for keypath, leaf in path_and_leaves if jnp.squeeze(np.asarray(leaf)).size == 1
    }
    return str_and_leaves


trainer_pl = Trainer(
    model=env_power_law,
    trainable_getter=power_law_env_getter,
    loss_fn=loss_fn,
    optimizer=optax.adabelief(3e-3),
    logger=WandbLogger(wandb.init(project="cmod_taue", entity="allen_adastra", group="power_law")),
)

eval_suite = {"error_plots": make_error_plots, "model_scalars": get_model_scalars}


trainer_pl.train(
    train_dl,
    val_dl,
    max_epochs=200,
    patience=50,
    epochs_per_val=1,
    eval_suite=eval_suite,
)

In [ ]:
# trainer_pl = Trainer(
#     model=env_power_law,
#     trainable_getter=power_law_env_getter,
#     loss_fn=loss_fn,
#     optimizer=optax.adabelief(1e-3),
#     logger=WandbLogger(wandb.init(project="cmod_taue", entity="allen_adastra", group="power_law")),
# )

# trainer_pl.train(
#     train_dl,
#     val_dl,
#     max_epochs=500,
#     patience=10,
#     epochs_per_val=5,
#     eval_suite=eval_suite,
# )

# trainer_nn = Trainer(
#     model=env_nn,
#     trainable_getter=nn_env_getter,
#     loss_fn=loss_fn,
#     optimizer=optax.adabelief(1e-3),
#     logger=WandbLogger(wandb.init(project="cmod_taue", entity="allen_adastra", group="nn")),
#     eval_suite=eval_suite,
# )
# trainer_nn.train(train_dl, val_dl, max_epochs=500, patience=10, epochs_per_val=5)